In [ ]:
from google.colab import userdata
import os
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')

In [3]:
!kaggle datasets download -d muthuj7/weather-dataset

Dataset URL: https://www.kaggle.com/datasets/muthuj7/weather-dataset
License(s): CC0-1.0
  0% 0.00/2.23M [00:00<?, ?B/s]
100% 2.23M/2.23M [00:00<00:00, 164MB/s]


In [4]:
! unzip "weather-dataset.zip"

Archive:  weather-dataset.zip
  inflating: weatherHistory.csv      


In [6]:
import pandas as pd

In [5]:
data=pd.read_csv("weatherHistory.csv", parse_dates=['Formatted Date'])

In [ ]:
data.head()

,Formatted Date,Summary,Precip Type,Temperature (C),Apparent Temperature (C),Humidity,Wind Speed (km/h),Wind Bearing (degrees),Visibility (km),Loud Cover,Pressure (millibars),Daily Summary
0,2006-04-01 00:00:00+02:00,Partly Cloudy,rain,9.472222,7.388889,0.89,14.1197,251.0,15.8263,0.0,1015.13,Partly cloudy throughout the day.
1,2006-04-01 01:00:00+02:00,Partly Cloudy,rain,9.355556,7.227778,0.86,14.2646,259.0,15.8263,0.0,1015.63,Partly cloudy throughout the day.
2,2006-04-01 02:00:00+02:00,Mostly Cloudy,rain,9.377778,9.377778,0.89,3.9284,204.0,14.9569,0.0,1015.94,Partly cloudy throughout the day.
3,2006-04-01 03:00:00+02:00,Partly Cloudy,rain,8.288889,5.944444,0.83,14.1036,269.0,15.8263,0.0,1016.41,Partly cloudy throughout the day.
4,2006-04-01 04:00:00+02:00,Mostly Cloudy,rain,8.755556,6.977778,0.83,11.0446,259.0,15.8263,0.0,1016.51,Partly cloudy throughout the day.


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96453 entries, 0 to 96452
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype              
---  ------                    --------------  -----              
 0   Formatted Date            96453 non-null  object             
 1   Summary                   96453 non-null  object             
 2   Precip Type               95936 non-null  object             
 3   Temperature (C)           96453 non-null  float64            
 4   Apparent Temperature (C)  96453 non-null  float64            
 5   Humidity                  96453 non-null  float64            
 6   Wind Speed (km/h)         96453 non-null  float64            
 7   Wind Bearing (degrees)    96453 non-null  float64            
 8   Visibility (km)           96453 non-null  float64            
 9   Loud Cover                96453 non-null  float64            
 10  Pressure (millibars)      96453 non-null  float64            
 11  Daily Summary  

In [9]:
data['Date']=pd.to_datetime(data['Formatted Date'],utc=True)

In [ ]:
data.Date.max()
# pay attention to the date format

Timestamp('2016-12-31 22:00:00+0000', tz='UTC')

- Extract the average temperature last week

In [ ]:
data[data['Date']>='2016-12-24'].groupby('Date')['Temperature (C)'].mean().mean()

0.26439790575916244

- Find the days on which the humidity exceed 75%

In [ ]:
data[data['Humidity']>0.75]['Date'].dt.date.drop_duplicates()

,Date
0,2006-03-31
2,2006-04-01
26,2006-04-10
50,2006-04-11
74,2006-04-12
...,...
96311,2016-09-04
96335,2016-09-05
96359,2016-09-06
96383,2016-09-07


- Compare the average wind speed on sunny days versus rainy days

In [ ]:
data.groupby('Summary')['Wind Speed (km/h)'].mean()

,Wind Speed (km/h)
Summary,
Breezy,32.143948
Breezy and Dry,33.810000
Breezy and Foggy,33.477880
Breezy and Mostly Cloudy,33.386345
Breezy and Overcast,33.037566
Breezy and Partly Cloudy,33.532796
Clear,8.141352
Dangerously Windy and Partly Cloudy,63.852600
Drizzle,10.356428


In [ ]:
data.groupby('Precip Type')['Wind Speed (km/h)'].mean()


,Wind Speed (km/h)
Precip Type,
rain,10.971219
snow,9.481998


# Gemini

In [10]:
# prompt: Find the average temperature for the last seven days from the most recent date in the dataset.

import pandas as pd
data[data['Date']>=data.Date.max()-pd.DateOffset(days=7)]['Temperature (C)'].mean()


0.22692307692307706

In [12]:
# prompt: prompt: Display and compare the average wind speed on days

data.groupby('Summary')['Wind Speed (km/h)'].mean()


,Wind Speed (km/h)
Summary,
Breezy,32.143948
Breezy and Dry,33.810000
Breezy and Foggy,33.477880
Breezy and Mostly Cloudy,33.386345
Breezy and Overcast,33.037566
Breezy and Partly Cloudy,33.532796
Clear,8.141352
Dangerously Windy and Partly Cloudy,63.852600
Drizzle,10.356428


In [13]:
# prompt: - List all the days on which humidity is greater than 75%, return their corresponding humidity levels.

data[data['Humidity'] > 0.75][['Date','Humidity']]


,Date,Humidity
0,2006-03-31 22:00:00+00:00,0.89
1,2006-03-31 23:00:00+00:00,0.86
2,2006-04-01 00:00:00+00:00,0.89
3,2006-04-01 01:00:00+00:00,0.83
4,2006-04-01 02:00:00+00:00,0.83
...,...,...
96432,2016-09-09 01:00:00+00:00,0.87
96433,2016-09-09 02:00:00+00:00,0.93
96434,2016-09-09 03:00:00+00:00,0.90
96435,2016-09-09 04:00:00+00:00,0.93


In [17]:
# prompt: Analyze data, how can i understand if day rainy or sunny

# Assuming 'Summary' column contains information about rain or sun

# Count occurrences of rainy and sunny days
rainy_days = data[data['Summary'].str.contains('Rain', case=False)].shape[0]

# Use '|' to specify multiple patterns for sunny days
sunny_days = data[data['Summary'].str.contains('Sun|Clear', case=False)].shape[0]

print(f"Rainy days: {rainy_days}")
print(f"Sunny days: {sunny_days}")

# Calculate percentage of rainy and sunny days
total_days = data.shape[0]
rainy_percentage = (rainy_days / total_days) * 100
sunny_percentage = (sunny_days / total_days) * 100

print(f"Rainy days percentage: {rainy_percentage:.2f}%")
print(f"Sunny days percentage: {sunny_percentage:.2f}%")

# Explore other columns related to rain or sun, like 'Precip Type'
rainy_precip_types = data[data['Summary'].str.contains('Rain', case=False)]['Precip Type'].unique()
print(f"Precipitation types on rainy days: {rainy_precip_types}")

Rainy days: 73
Sunny days: 10890
Rainy days percentage: 0.08%
Sunny days percentage: 11.29%
Precipitation types on rainy days: ['rain']
